# Ablação Linear — DEPREL + HEAD sem UPOS

Treina os 4 modelos com os melhores hiperparâmetros do Optuna (`cv_results_*_linear.csv`) mas **sem a cabeça de UPOS**, para avaliar o impacto do POS tagging auxiliar sobre UAS/LAS.

**Modelos:** mBERT · BERTimbau-Base · BERTimbau-Large · ModernJabuticaBERT

In [1]:
import os, random, numpy as np, torch

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    print(f"Seed definida como {seed}")

seed_everything(42)

Seed definida como 42


In [2]:
import numpy as np
import os, json, gc, shutil
from datetime import datetime
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
from typing import Optional

import datasets
from datasets import Dataset, DatasetDict, load_from_disk

from transformers import (
    AutoTokenizer, AutoConfig, AutoModel,
    BertPreTrainedModel,
    Trainer, TrainingArguments, EarlyStoppingCallback,
)

/home/guilhermelima/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Labels

In [3]:
DEPREL_LABELS = [
    'det', 'nsubj', 'root', 'obj', 'xcomp', 'punct', 'mark', 'advcl', 'case',
    'obl', 'amod', 'conj', 'cc', 'nmod', 'advmod', 'flat:name', 'ccomp', 'cop',
    'acl', 'nummod', 'acl:relcl', 'ccomp:speech', 'parataxis', 'csubj',
    'aux:pass', 'appos', 'fixed', 'nsubj:pass', 'aux', 'nsubj:outer',
    'obl:agent', 'expl:impers', 'expl', 'discourse', 'orphan', 'dislocated',
    'flat', 'flat:foreign', 'iobj', 'vocative', 'csubj:outer', 'list',
    'reparandum', 'csubj:pass',
]

DEPREL_LABELS_TO_IDX = {l: i for i, l in enumerate(DEPREL_LABELS)}
IDX_TO_DEPREL_LABELS = {i: l for l, i in DEPREL_LABELS_TO_IDX.items()}

print(f"DEPREL labels: {len(DEPREL_LABELS)}")

DEPREL labels: 44


## Melhores hiperparâmetros (Optuna linear)

In [4]:
# Linha 0 de cada cv_results_*_linear.csv = melhor média de LAS no k-fold
BEST_HPS_LINEAR = {
    #"google-bert/bert-base-multilingual-cased": {
    #    "learning_rate":   3.625756168886472e-05,
    #    "weight_decay":    0.19673026818206146,
    #    "warmup_ratio":    0.42050513822312574,
    #    "num_train_epochs": 40,
    #},
    "neuralmind/bert-base-portuguese-cased": {
        "learning_rate":   4.423627211584912e-05,
        "weight_decay":    0.2143062841617503,
        "warmup_ratio":    0.36745951295566587,
        "num_train_epochs": 40,
    },
    #"neuralmind/bert-large-portuguese-cased": {
    #    "learning_rate":   3.755373147243102e-05,
    #    "weight_decay":    0.12615267832325944,
    #    "warmup_ratio":    0.4036975194086722,
    #    "num_train_epochs": 40,
    #},
    #"amadeusai/modernJabuticaBERT-Base-1k": {
    #    "learning_rate":   4.9263783534529467e-05,
    #    "weight_decay":    0.1496766408078372,
    #    "warmup_ratio":    0.43543301938527523,
    #    "num_train_epochs": 40,
    #},
}

MODELS = list(BEST_HPS_LINEAR.keys())
print("Modelos:", MODELS)

Modelos: ['neuralmind/bert-base-portuguese-cased']


## Carregamento dos dados

In [5]:
import ast

def load_csv_as_hf_dataset(filepath):
    df = pd.read_csv(filepath)
    records = []
    for _, row in df.iterrows():
        records.append({
            'tokens':     ast.literal_eval(row['tokens']),
            'upos':       ast.literal_eval(row['upos']),
            'deprel':     ast.literal_eval(row['deprel']),
            'head_tags':  ast.literal_eval(str(row['head_tags'])),
            'deprel_tags':ast.literal_eval(str(row['deprel_tags'])),
            'upos_tags':  ast.literal_eval(str(row['upos_tags'])),
        })
    return Dataset.from_list(records)

data = DatasetDict({
    'train': load_csv_as_hf_dataset('/home/guilhermelima/msc/data_dois/train_outxpos.csv'),
    'val':   load_csv_as_hf_dataset('/home/guilhermelima/msc/data_dois/val_outxpos.csv'),
    'test':  load_csv_as_hf_dataset('/home/guilhermelima/msc/data_dois/test_outxpos.csv'),
})
data

DatasetDict({
    train: Dataset({
        features: ['tokens', 'upos', 'deprel', 'head_tags', 'deprel_tags', 'upos_tags'],
        num_rows: 5893
    })
    val: Dataset({
        features: ['tokens', 'upos', 'deprel', 'head_tags', 'deprel_tags', 'upos_tags'],
        num_rows: 842
    })
    test: Dataset({
        features: ['tokens', 'upos', 'deprel', 'head_tags', 'deprel_tags', 'upos_tags'],
        num_rows: 1683
    })
})

## Modelo — Linear sem UPOS

In [6]:
# Ablação: apenas DEPREL + HEAD, sem cabeça de UPOS
class MultiTaskSentencePredictionEncoderAblacao(BertPreTrainedModel):
    _tied_weights_keys = []
    all_tied_weights_keys = {}

    def __init__(self, config, num_deprel_labels, num_head_labels=200):
        super().__init__(config)
        self.num_deprel_labels = num_deprel_labels
        self.num_head_labels   = num_head_labels

        self.bert = AutoModel.from_config(config)

        self.deprel_classifier = nn.Linear(config.hidden_size, num_deprel_labels)
        self.head_classifier   = nn.Linear(config.hidden_size, num_head_labels)

        classifier_dropout = (
            getattr(config, 'classifier_dropout', None)
            or getattr(config, 'hidden_dropout_prob', 0.1)
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.init_weights()

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        token_type_ids: Optional[torch.Tensor] = None,
        deprel_label:   Optional[torch.Tensor] = None,
        head_label:     Optional[torch.Tensor] = None,
    ):
        try:
            outputs = self.bert(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
        except TypeError:
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        sequence_output = self.dropout(outputs[0])

        logits_deprel = self.deprel_classifier(sequence_output)
        logits_head   = self.head_classifier(sequence_output)

        loss = None
        if deprel_label is not None and head_label is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = (
                loss_fct(logits_deprel.view(-1, self.num_deprel_labels), deprel_label.view(-1))
                + loss_fct(logits_head.view(-1, self.num_head_labels),   head_label.view(-1))
            )

        if loss is not None:
            return (loss, logits_deprel, logits_head)
        return (logits_deprel, logits_head)

## POSDataset — sem UPOS

In [7]:
class POSDataset:
    def __init__(self, tokenizer_ckpt):
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_ckpt)

    def align_labels_with_tokens(self, labels, word_ids):
        new_labels, current_word = [], None
        for word_id in word_ids:
            if word_id != current_word:
                current_word = word_id
                try:
                    label = -100 if word_id is None else labels[word_id]
                except Exception:
                    label = -100
                new_labels.append(label)
            elif word_id is None:
                new_labels.append(-100)
            else:
                new_labels.append(labels[word_id])
        return new_labels

    def preprocess_function(self, examples):
        tokenized_inputs = self.tokenizer(
            examples["tokens"],
            truncation=True,
            padding="max_length",
            is_split_into_words=True,
            max_length=512,
        )
        all_deprel = examples["deprel_tags"]
        all_head   = examples["head_tags"]
        new_deprel, new_head = [], []
        for i, (dep, head) in enumerate(zip(all_deprel, all_head)):
            word_ids = tokenized_inputs.word_ids(i)
            new_deprel.append(self.align_labels_with_tokens(dep,  word_ids))
            new_head.append(  self.align_labels_with_tokens(head, word_ids))
        tokenized_inputs["deprel_label"] = new_deprel
        tokenized_inputs["head_label"]   = new_head
        return tokenized_inputs

    def create_data(self, train, test):
        tkn_train = train.map(self.preprocess_function, batched=True, remove_columns=train.column_names)
        tkn_test  = test.map( self.preprocess_function, batched=True, remove_columns=test.column_names)
        return tkn_train, tkn_test

## Data Collator

In [8]:
def data_collator(batch):
    input_ids       = [item["input_ids"]       for item in batch]
    attention_masks = [item["attention_mask"]   for item in batch]
    deprel_label    = [item["deprel_label"]     for item in batch]
    head_label      = [item["head_label"]       for item in batch]

    max_len = max(len(ids) for ids in input_ids)
    PAD = 0

    return {
        "input_ids":      torch.tensor([ids + [PAD]  * (max_len - len(ids))  for ids  in input_ids]),
        "attention_mask": torch.tensor([m   + [0]    * (max_len - len(m))    for m    in attention_masks]),
        "deprel_label":   torch.tensor([l   + [-100] * (max_len - len(l))    for l    in deprel_label]),
        "head_label":     torch.tensor([l   + [-100] * (max_len - len(l))    for l    in head_label]),
    }

## Compute Metrics

In [9]:
import numpy as np
import json
import os

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)


def compute_metrics(eval_pred, PRETRAINED_MODEL=None):
    save_path = f"epoch_predictions/{PRETRAINED_MODEL.split('/')[-1] if PRETRAINED_MODEL else 'unknown_model'}_predictions_ablacao_linear.json"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    model_name = PRETRAINED_MODEL.split("/")[-1] if PRETRAINED_MODEL else "unknown_model"
    run_id = model_name

    logits, labels = eval_pred
    deprel_logits, head_logits = logits
    deprel_labels, head_labels = labels

    # ---------------- PROBABILIDADES ----------------
    deprel_probs = softmax(deprel_logits, axis=-1)
    head_probs   = softmax(head_logits,   axis=-1)

    # ---------------- PREDIÇÕES ----------------
    deprel_preds = np.argmax(deprel_logits, axis=-1)
    head_preds   = np.argmax(head_logits,   axis=-1)

    # ---------------- MASK DEPENDENCY ----------------
    valid_mask = (head_labels != -100) & (deprel_labels != -100)

    head_preds_masked    = head_preds[valid_mask]
    head_labels_masked   = head_labels[valid_mask]
    head_probs_masked    = head_probs[valid_mask]
    deprel_preds_masked  = deprel_preds[valid_mask]
    deprel_labels_masked = deprel_labels[valid_mask]
    deprel_probs_masked  = deprel_probs[valid_mask]

    # ---------------- MÉTRICAS PRINCIPAIS ----------------
    uas = (head_preds_masked == head_labels_masked).mean()
    las = (
        (head_preds_masked == head_labels_masked) &
        (deprel_preds_masked == deprel_labels_masked)
    ).mean()

    # ---------------- CARREGAR JSON ----------------
    if os.path.exists(save_path):
        with open(save_path, "r", encoding="utf-8") as f:
            all_data = json.load(f)
    else:
        all_data = {}

    if "META" not in all_data:
        all_data["META"] = {
            "model_name": model_name,
            "pretrained_model": PRETRAINED_MODEL,
            "run_id": run_id,
        }

    if "RANK_PREDICTIONS" not in all_data:
        all_data["RANK_PREDICTIONS"] = {"deprel": [], "head": []}

    def get_correct_rank(prob_vector, correct_label):
        sorted_indices = np.argsort(prob_vector)[::-1]
        return int(np.where(sorted_indices == correct_label)[0][0] + 1)

    # ---------------- DEPREL ----------------
    for i in range(len(deprel_preds_masked)):
        probs         = deprel_probs_masked[i]
        correct_label = int(deprel_labels_masked[i])
        pred_label    = int(deprel_preds_masked[i])
        all_data["RANK_PREDICTIONS"]["deprel"].append({
            "run_id":        run_id,
            "model":         model_name,
            "index":         i,
            "correct_label": correct_label,
            "pred_label":    pred_label,
            "correct_prob":  float(probs[correct_label]),
            "pred_prob":     float(probs[pred_label]),
            "correct_rank":  get_correct_rank(probs, correct_label),
        })

    # ---------------- HEAD ----------------
    for i in range(len(head_preds_masked)):
        probs         = head_probs_masked[i]
        correct_label = int(head_labels_masked[i])
        pred_label    = int(head_preds_masked[i])
        all_data["RANK_PREDICTIONS"]["head"].append({
            "run_id":        run_id,
            "model":         model_name,
            "index":         i,
            "correct_label": correct_label,
            "pred_label":    pred_label,
            "correct_prob":  float(probs[correct_label]),
            "pred_prob":     float(probs[pred_label]),
            "correct_rank":  get_correct_rank(probs, correct_label),
        })

    # ---------------- SALVAR ----------------
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(all_data, f, indent=2, ensure_ascii=False)

    return {"uas": float(uas), "las": float(las)}


## CUDA diagnóstico

In [10]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

if torch.cuda.is_available():
    t = torch.tensor([1.0], device="cuda")
    assert (t * 2).item() == 2.0
    del t
    torch.cuda.empty_cache()
    print("✓ Contexto CUDA limpo")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CPU mode")

✓ Contexto CUDA limpo
GPU: NVIDIA GeForce RTX 4090


## Wandb

In [11]:
os.environ["WANDB_API_KEY"] = "7499f50c2fffad3ce3e34b7b7b02152988a1742c"
import wandb

## Função de treino com HPs fixos (train/val original)

In [12]:
def train_ablacao_linear(name_model, best_hps, results_filename="ablacao_linear_results.jsonl"):
    learning_rate    = best_hps["learning_rate"]
    weight_decay     = best_hps["weight_decay"]
    warmup_ratio     = best_hps["warmup_ratio"]
    num_train_epochs = best_hps["num_train_epochs"]

    nerdataset = POSDataset(name_model)
    train_data, valid_data = nerdataset.create_data(data['train'], data['val'])

    print(f"\n{'='*55}")
    print(f"  {name_model}")
    print(f"{'='*55}")

    wandb.init(
        entity="gdlima-universidade-federal-de-pelotas",
        project="hf-optuna",
        name=f"ablacao_linear_{name_model.split('/')[-1]}",
        config={
            "learning_rate": learning_rate, "architecture": name_model,
            "epochs": num_train_epochs, "weight_decay": weight_decay,
            "warmup_ratio": warmup_ratio, "ablation": "sem_upos_linear",
        },
    )

    config = AutoConfig.from_pretrained(name_model)
    model  = MultiTaskSentencePredictionEncoderAblacao.from_pretrained(
        name_model, config=config,
        num_deprel_labels=len(DEPREL_LABELS),
        _fast_init=False,
    )

    for _n, _p in model.named_parameters():
        if _p.requires_grad and (torch.isnan(_p).any() or torch.isinf(_p).any()):
            raise RuntimeError(f"[CPU] Peso '{_n}' é NaN/Inf antes de mover para CUDA.")

    _device = "cuda" if torch.cuda.is_available() else "cpu"
    model   = model.to(_device)

    output_dir = f"./ablacao_linear_{name_model.replace('/','_')}"

    training_args = TrainingArguments(
        output_dir=output_dir,
        fp16=False, bf16=False,
        eval_strategy="epoch",
        learning_rate=learning_rate,
        num_train_epochs=num_train_epochs,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        max_grad_norm=1.0,
        lr_scheduler_type="linear",
        logging_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        save_only_model=True,
        load_best_model_at_end=True,
        metric_for_best_model="las",
        greater_is_better=True,
        label_smoothing_factor=0.0,
        gradient_checkpointing=False,
        remove_unused_columns=False,
        label_names=["deprel_label", "head_label"],
        report_to="wandb",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        dataloader_num_workers=4,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=valid_data,
        compute_metrics=lambda p: compute_metrics(p, PRETRAINED_MODEL=name_model),
        data_collator=data_collator,
        #callbacks=[EarlyStoppingCallback(5)],
    )

    trainer.train()

    # --- salvar log por época ---
    train_logs = {
        int(e["epoch"]): e["loss"]
        for e in trainer.state.log_history
        if "loss" in e and "eval_loss" not in e
    }
    eval_logs = {
        int(e["epoch"]): e
        for e in trainer.state.log_history
        if "eval_loss" in e
    }
    epoch_rows = []
    for epoch in sorted(set(train_logs) | set(eval_logs)):
        epoch_rows.append({
            "Epoch":           epoch,
            "Training Loss":   train_logs.get(epoch, None),
            "Validation Loss": eval_logs[epoch]["eval_loss"] if epoch in eval_logs else None,
            "Uas":             eval_logs[epoch].get("eval_uas", None) if epoch in eval_logs else None,
            "Las":             eval_logs[epoch].get("eval_las", None) if epoch in eval_logs else None,
        })
    log_df = pd.DataFrame(epoch_rows)
    log_csv = f"training_log_ablacao_linear_{name_model.replace('/','_')}.csv"
    log_df.to_csv(log_csv, index=False)
    print(f"  Log por época salvo em: {log_csv}")
    # --- fim do log ---

    best_path = f"./best_models_ablacao_linear/{name_model.replace('/','_')}"
    trainer.save_model(best_path)
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    eval_result = trainer.evaluate()
    uas = eval_result.get("eval_uas", None)
    las = eval_result.get("eval_las", None)

    result_dict = {
        "name": name_model,
        "architecture": "linear_ablacao_sem_upos",
        "hyperparameters": best_hps,
        "uas": uas, "las": las,
    }

    with open(results_filename, "a", encoding="utf-8") as f:
        f.write(json.dumps(result_dict, ensure_ascii=False) + "\n")

    print(f"  UAS: {uas:.4f} | LAS: {las:.4f}")
    wandb.finish()
    return result_dict


## Treino — todos os modelos

In [13]:
all_results = {}

for model_name in MODELS:
    print(f"\n{'#'*60}")
    print(f"# ABLAÇÃO LINEAR (sem UPOS) — {model_name}")
    print(f"{'#'*60}")
    all_results[model_name] = train_ablacao_linear(model_name, BEST_HPS_LINEAR[model_name])



############################################################
# ABLAÇÃO LINEAR (sem UPOS) — neuralmind/bert-base-portuguese-cased
############################################################


Map: 100%|██████████| 842/842 [00:00<00:00, 2191.77 examples/s]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.



  neuralmind/bert-base-portuguese-cased


wandb: Currently logged in as: gdlima (gdlima-universidade-federal-de-pelotas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 32789.88it/s]
[transformers] MultiTaskSentencePredictionEncoderAblacao LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
head_classifier.bias                       | MISSING    | 
head_classifier.weight                     | MISSING    | 
deprel_classifier.bias                     | MISSING    | 
deprel_classifier.weight                   | MISSI

...

  Log por época salvo em: training_log_ablacao_linear_neuralmind_bert-base-portuguese-cased.csv


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.06it/s]


Training Loss,Validation Loss,Epoch,Uas,Las
0.010815,0.751873,40,0.929778,0.916420


  UAS: 0.9298 | LAS: 0.9164


eval/las,▁▂▃▄▅▅▆▆▇▇▇▇▇▇▇▇████████████████████████
eval/loss,█▅▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/runtime,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇█
eval/samples_per_second,█▇▇▆▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
eval/steps_per_second,█▇▇▆▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
eval/uas,▁▂▃▄▅▅▆▆▇▇▇▇▇▇▇▇████████████████████████
train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇██
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▇▇▇▇▇▇▇███
train/grad_norm,▂▂▃▃▃▃▄▄▅▄▅▃▄█▃▆▃▂▆▄▃▂▂▃▄▁▂▁▂▇▁▁▂▁▂▂▁▃▂▁
train/learning_rate,▁▂▂▃▃▄▄▅▅▆▆▇▇███▇▇▇▇▆▆▆▅▅▅▅▄▄▄▄▃▃▃▂▂▂▂▁▁
+1,...


## Inferência no conjunto de teste

In [14]:
dataset_test = load_from_disk('/home/guilhermelima/msc/data_dois/complaints_dataset_obj_outxpos')
test_dataset  = dataset_test['test']
test_sentences = test_dataset['tokens']
test_deprel    = test_dataset['deprel']
test_head      = test_dataset['head_tags']

print(f"Test set: {len(test_sentences)} sentenças")
print(f"Exemplo: {test_sentences[0]}")

Test set: 1683 sentenças
Exemplo: ['O', 'Capitão', 'América', 'também', 'bajulou', 'o', 'tucano', '.']


In [15]:
def get_predictions_on_dataframe(sentences, model, tokenizer, device="cuda"):
    predictions_deprel, predictions_head = [], []

    model.eval()
    model.to(device)

    for tokens in tqdm(sentences):
        inputs = tokenizer(
            tokens, is_split_into_words=True,
            return_tensors="pt", padding=True, truncation=True,
        ).to(device)

        with torch.no_grad():
            model_outputs = model(**inputs)

        logits_deprel = model_outputs[0]
        logits_head   = model_outputs[1]
        word_ids = inputs.word_ids(batch_index=0)

        sent_deprel, sent_head = [], []
        for token_idx in range(len(tokens)):
            sub_idxs = [i for i, w in enumerate(word_ids) if w == token_idx]
            if sub_idxs:
                first = sub_idxs[0]
                prob_d = torch.softmax(logits_deprel[0, first], dim=-1)
                prob_h = torch.softmax(logits_head[0, first],   dim=-1)
                sent_deprel.append(IDX_TO_DEPREL_LABELS[torch.argmax(prob_d).item()])
                sent_head.append(torch.argmax(prob_h).item())

        predictions_deprel.append(sent_deprel)
        predictions_head.append(sent_head)

    return pd.DataFrame({
        "tokens":             sentences,
        "deprel_predictions": predictions_deprel,
        "head_predictions":   predictions_head,
    })

In [16]:
def compute_dependency_metrics(test_sentences, test_deprel, test_head, predict_df):
    total = uas_correct = las_correct = skipped = 0

    for i in range(len(test_sentences)):
        gold_d = test_deprel[i]
        gold_h = test_head[i]
        pred_d = predict_df['deprel_predictions'].iloc[i]
        pred_h = predict_df['head_predictions'].iloc[i]

        for j in range(len(gold_h)):
            if j >= len(pred_h) or pred_h[j] is None or pred_d[j] is None:
                skipped += 1
                continue
            total += 1
            if pred_h[j] == gold_h[j]:
                uas_correct += 1
                if pred_d[j] == gold_d[j]:
                    las_correct += 1

    return {
        'uas':          uas_correct / total if total > 0 else 0,
        'las':          las_correct / total if total > 0 else 0,
        'total_tokens': total,
        'skipped':      skipped,
    }

In [17]:
_device = "cuda" if torch.cuda.is_available() else "cpu"
final_metrics = {}

for model_name in MODELS:
    print(f"\n{'='*50}")
    print(f"Inferência: {model_name}")

    best_model_path = f"./best_models_ablacao_linear/{model_name.replace('/','_')}"
    val_las = all_results[model_name]["las"]
    print(f"Modelo salvo em: {best_model_path} | LAS val = {val_las:.4f}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    config    = AutoConfig.from_pretrained(best_model_path)
    model     = MultiTaskSentencePredictionEncoderAblacao.from_pretrained(
        best_model_path, config=config,
        num_deprel_labels=len(DEPREL_LABELS),
    ).to(_device)

    predict_df = get_predictions_on_dataframe(test_sentences, model, tokenizer, device=_device)
    metrics    = compute_dependency_metrics(test_sentences, test_deprel, test_head, predict_df)
    final_metrics[model_name] = metrics

    print(f"  UAS: {metrics['uas']:.4f}")
    print(f"  LAS: {metrics['las']:.4f}")
    print(f"  Total tokens: {metrics['total_tokens']} | Ignorados: {metrics['skipped']}")

    out_csv = f"./predict_test_ablacao_linear_{model_name.replace('/','_')}.csv"
    predict_df.to_csv(out_csv, index=False)
    print(f"  Salvo em: {out_csv}")



Inferência: neuralmind/bert-base-portuguese-cased
Modelo salvo em: ./best_models_ablacao_linear/neuralmind_bert-base-portuguese-cased | LAS val = 0.9164


100%|██████████| 1683/1683 [00:13<00:00, 126.66it/s]


  UAS: 0.9352
  LAS: 0.9223
  Total tokens: 33580 | Ignorados: 0
  Salvo em: ./predict_test_ablacao_linear_neuralmind_bert-base-portuguese-cased.csv


## Resumo final

In [ ]:
print("\n" + "="*65)
print("ABLAÇÃO LINEAR — DEPREL + HEAD (sem UPOS) — Conjunto de Teste")
print("="*65)
print(f"{'Modelo':<42} {'UAS':>7} {'LAS':>7}")
print("-"*65)
for model_name, metrics in final_metrics.items():
    short = model_name.split('/')[-1]
    print(f"{short:<42} {metrics['uas']:>7.4f} {metrics['las']:>7.4f}")

# Salvar métricas finais
results_summary = [
    {"model": k, "uas": v["uas"], "las": v["las"],
     "total_tokens": v["total_tokens"], "architecture": "linear_ablacao_sem_upos"}
    for k, v in final_metrics.items()
]
pd.DataFrame(results_summary).to_csv("ablacao_linear_test_metrics.csv", index=False)
print("\nMétricas salvas em ablacao_linear_test_metrics.csv")


ABLAÇÃO LINEAR — DEPREL + HEAD (sem UPOS) — Conjunto de Teste
Modelo                                         UAS     LAS
-----------------------------------------------------------------
bert-base-portuguese-cased                  0.9352  0.9223

Métricas salvas em ablacao_linear_test_metrics.csv


: 